In [1]:
from conformer.tokenizer import Tokenizer
from pathlib import Path
from conformer.config import DataConfig, TrainingConfig
from conformer.model import ConformerEncoder
from flax import  nnx
import optax
import grain
from conformer.dataset import batch_fn, ProcessAudioData, unpack_speech_data
import jax

In [2]:
data_config = DataConfig()
train_config = TrainingConfig() 

In [3]:
tokenizer = Tokenizer.load_tokenizer(Path(data_config.tokenizer_path))

In [4]:
model = ConformerEncoder(token_count=len(tokenizer.id_to_char))

W0212 21:18:57.773850   30709 cuda_executor.cc:1802] GPU interconnect information not available: INTERNAL: NVML doesn't support extracting fabric info or NVLink is not used by the device.
W0212 21:18:57.776749   30634 cuda_executor.cc:1802] GPU interconnect information not available: INTERNAL: NVML doesn't support extracting fabric info or NVLink is not used by the device.


In [5]:
lr_schedule = optax.warmup_cosine_decay_schedule(
    init_value=train_config.lr_init_value,
    peak_value=train_config.lr_peak_value,
    warmup_steps=train_config.lr_warmup_steps,
    decay_steps=train_config.lr_decay_steps,
    end_value=train_config.lr_end_value
)

In [6]:
optimizer = nnx.Optimizer(
    model,
    optax.adamw(
        learning_rate=lr_schedule,
        b1=0.9,
        b2=0.98,
        weight_decay=1e-2,
        mask=lambda p: jax.tree.map(lambda x: x.ndim > 1, p)
    ),
    wrt=nnx.Param
)

In [7]:
train_audio_source = grain.sources.ArrayRecordDataSource(data_config.train_data_path)
test_audio_source = grain.sources.ArrayRecordDataSource(data_config.test_data_path)


map_train_audio_dataset = grain.MapDataset.source(train_audio_source)
map_test_audio_dataset = grain.MapDataset.source(test_audio_source)


processed_train_dataset = (
    map_train_audio_dataset
    .shuffle(seed=42)
    .map(ProcessAudioData(tokenizer))
    .batch(batch_size=data_config.batch_size, batch_fn=batch_fn)
)

In [8]:
processed_test_dataset = (
    map_test_audio_dataset
    .map(ProcessAudioData(tokenizer))
    .batch(batch_size=data_config.batch_size, batch_fn=batch_fn)
)


In [14]:
@nnx.jit
def jitted_train(model, optimizer, padded_audios, padded_labels, frames, label_lengths):
    """Training step with gradient computation"""
    def loss_fn(model):
        logits, real_times = model(padded_audios, training=True, inputs_lengths=frames)
        
        logit_paddings = (jnp.arange(logits.shape[1]) >= real_times[:, None]).astype(jnp.float32)
        label_paddings = (jnp.arange(padded_labels.shape[1]) >= label_lengths[:, None]).astype(jnp.float32)
        
        loss = optax.ctc_loss(logits, logit_paddings, padded_labels, label_paddings, blank_id=tokenizer.blank_id).mean()
        return loss
    
    loss, grads = nnx.value_and_grad(loss_fn)(model)
    optimizer.update(model=model, grads=grads)
    return loss

In [15]:
warmup_batch = processed_train_dataset[0]
padded_audios, frames, padded_labels, label_lengths = warmup_batch

In [16]:
logits, real_times = model(padded_audios, mask=None, training=True, inputs_lengths=frames)

In [17]:
import jax.numpy as jnp

In [21]:
jitted_train(model, optimizer, padded_audios, padded_labels, frames, label_lengths)

Array(381.5229, dtype=float32)

In [23]:
from tqdm.auto import  tqdm

In [26]:
epoch = 0

In [30]:
train_loss_sum = 0.0
train_steps = 0
global_step = 0

In [ ]:
pbar = tqdm(processed_train_dataset, desc=f"Epoch {epoch+1}/{train_config.num_epochs}")
for element in pbar:
    padded_audios, frames, padded_labels, label_lengths = element
    loss = jitted_train(model, optimizer, padded_audios, padded_labels, frames, label_lengths)
    
    train_loss_sum += float(loss)
    train_steps += 1
    global_step += 1
    
    # Update tqdm with running average loss
    avg_train_loss = train_loss_sum / train_steps
    pbar.set_postfix({"train_loss": f"{avg_train_loss:.2f}", "step": global_step})

Epoch 1/1:   0%|          | 0/5255 [00:00<?, ?it/s]

In [ ]:
import jax
import jax.numpy as jnp

In [ ]:
signal = jax.random.uniform(jax.random.key(42), (4, 16_000 * 6))
lengths = jnp.array([16_000 * 3, 16_000 * 4, 16_000 * 5, 16_000 * 6] )

In [ ]:
mel, l = model.mel_spectogram(signal, lengths)

In [ ]:
logits, seq_len = model(signal, inputs_lengths=lengths)

In [ ]:
seq_len